In [1]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
import PIL
from tensorflow.keras.preprocessing.image import img_to_array , load_img
import pickle 
import gzip 
import logging 
# tf.get_logger().setLevel(logging.ERROR)

In [2]:
TRAINING_FILE_DIR = './coco_dataset/train2014/'
OUTPUT_FILE_DIR = 'tf_data/feature_vectors/'

In [3]:
with open(TRAINING_FILE_DIR \
          + 'captions_train2014.json') as json_file:
    data = json.load(json_file)
image_dict = {}
for image in data['images']:
    image_dict[image['id']] = [image['file_name']]
for anno in data['annotations']:
    image_dict[anno['image_id']].append(anno['caption'])

In [4]:
image_dict

{57870: ['COCO_train2014_000000057870.jpg',
  'A restaurant has modern wooden tables and chairs.',
  'A long restaurant table with rattan rounded back chairs.',
  'a long table with a plant on top of it surrounded with wooden chairs ',
  'A long table with a flower arrangement in the middle for meetings',
  'A table is adorned with wooden chairs with blue accents.'],
 384029: ['COCO_train2014_000000384029.jpg',
  'A man preparing desserts in a kitchen covered in frosting.',
  'A chef is preparing and decorating many small pastries.',
  'A baker prepares various types of baked goods.',
  'a close up of a person grabbing a pastry in a container',
  'Close up of a hand touching various pastries.'],
 222016: ['COCO_train2014_000000222016.jpg',
  'a big red telephone booth that a man is standing in',
  'a person standing inside of a phone booth ',
  'this is an image of a man in a phone booth.',
  'A man is standing in a red phone booth.',
  'A man using a phone in a phone booth.'],
 520950

In [5]:
#Model
model = VGG19(weights='imagenet')
model.summary()

Instructions for updating:
If using Keras pass *_constraint arguments to layers.


Model: "vgg19"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 224, 224, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 224, 224, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 112, 112, 64)      0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 112, 112, 128)     73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 112, 112, 128)     147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, 56, 56, 128)       0     

In [6]:
model_new = Model(inputs=model.input,
                  outputs=model.get_layer('block5_conv4').output)
model_new.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 224, 224, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 224, 224, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 112, 112, 64)      0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 112, 112, 128)     73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 112, 112, 128)     147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, 56, 56, 128)       0     

In [7]:
for i, key in enumerate(image_dict.keys()):
    if i % 1000 == 0:
        print('Progress: ' + str(i) + ' images processed')
    item = image_dict.get(key)
    filename = TRAINING_FILE_DIR + 'train2014/' + item[0]

    # Determine dimensions.
    image = load_img(filename)
    width = image.size[0]
    height = image.size[1]

    # Resize so shortest side is 256 pixels.
    if height > width:
        image = load_img(filename, target_size=(
            int(height/width*256), 256))
    else:
        image = load_img(filename, target_size=(
            256, int(width/height*256)))
    width = image.size[0]
    height = image.size[1]
    image_np = img_to_array(image)

    # Crop to center 224x224 region.
    h_start = int((height-224)/2)
    w_start = int((width-224)/2)
    image_np = image_np[h_start:h_start+224,
                        w_start:w_start+224]

    # Rearrange array to have one more
    # dimension representing batch size = 1.
    image_np = np.expand_dims(image_np, axis=0)

    # Call model and save resulting tensor to disk.
    X = preprocess_input(image_np)
    y = model_new.predict(X)
    save_filename = OUTPUT_FILE_DIR + \
        item[0] + '.pickle.gzip'
    pickle_file = gzip.open(save_filename, 'wb')
    pickle.dump(y[0], pickle_file)
    pickle_file.close()


Progress: 0 images processed
Progress: 1000 images processed
Progress: 2000 images processed
Progress: 3000 images processed
Progress: 4000 images processed
Progress: 5000 images processed
Progress: 6000 images processed
Progress: 7000 images processed
Progress: 8000 images processed
Progress: 9000 images processed
Progress: 10000 images processed
Progress: 11000 images processed
Progress: 12000 images processed
Progress: 13000 images processed
Progress: 14000 images processed
Progress: 15000 images processed
Progress: 16000 images processed
Progress: 17000 images processed
Progress: 18000 images processed
Progress: 19000 images processed
Progress: 20000 images processed
Progress: 21000 images processed
Progress: 22000 images processed
Progress: 23000 images processed
Progress: 24000 images processed
Progress: 25000 images processed
Progress: 26000 images processed
Progress: 27000 images processed
Progress: 28000 images processed
Progress: 29000 images processed
Progress: 30000 images 

In [8]:
#Saving filename and caption data
save_filename = OUTPUT_FILE_DIR + 'caption_file.pickle.gz'
pickle_file = gzip.open(save_filename, 'wb')
pickle.dump(image_dict, pickle_file)
pickle_file.close()